# EduDocAI — Exploratory Data Analysis

This notebook performs EDA on the sample documents and evaluation dataset.
Run after indexing: `python scripts/prepare_dataset.py`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from config.settings import settings, SAMPLE_DOCS_DIR, EVAL_DIR
from src.ingestion.document_loader import DocumentLoader
from src.preprocessing.chunking import TextChunker
print('Imports OK')

## 1. Load Sample Documents

In [ ]:
loader = DocumentLoader()
chunker = TextChunker()

all_pages = []
doc_stats = []

for f in sorted(SAMPLE_DOCS_DIR.glob('*.pdf')):
    pages = loader.load_from_path(f)
    chunks = chunker.chunk_documents(pages)
    all_pages.extend(pages)
    total_words = sum(c.word_count for c in chunks)
    doc_stats.append({
        'Document': f.name,
        'Pages': len(pages),
        'Chunks': len(chunks),
        'Total Words': total_words,
        'Avg Words/Page': round(total_words / max(len(pages), 1), 1)
    })
    print(f'{f.name}: {len(pages)} pages, {len(chunks)} chunks, {total_words:,} words')

df_docs = pd.DataFrame(doc_stats)
df_docs

## 2. Document Text-Length Distribution

In [ ]:
page_lengths = [len(p.text.split()) for p in all_pages if p.text.strip()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(page_lengths, bins=20, color='#2563EB', alpha=0.75, edgecolor='white')
axes[0].set_xlabel('Words per Page')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Page Word-Count Distribution')
axes[0].spines[['top', 'right']].set_visible(False)

by_doc = {}
for p in all_pages:
    by_doc.setdefault(p.source, []).append(len(p.text.split()))
labels = list(by_doc.keys())
values = [sum(v) for v in by_doc.values()]
axes[1].barh(labels, values, color='#2563EB', alpha=0.8)
axes[1].set_xlabel('Total Words')
axes[1].set_title('Total Words per Document')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()
print(f'Mean words/page: {np.mean(page_lengths):.1f}')
print(f'Std: {np.std(page_lengths):.1f}')

## 3. Chunk-Length Distribution

In [ ]:
all_chunks = chunker.chunk_documents(all_pages)
chunk_words = [c.word_count for c in all_chunks]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(chunk_words, bins=30, color='#2563EB', alpha=0.75, edgecolor='white')
ax.axvline(np.mean(chunk_words), color='red', linestyle='--', label=f'Mean={np.mean(chunk_words):.0f}')
ax.set_xlabel('Words per Chunk')
ax.set_ylabel('Count')
ax.set_title('Chunk Word-Count Distribution (chunk_size=400, overlap=60)')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Total chunks : {len(all_chunks)}')
print(f'Mean words   : {np.mean(chunk_words):.1f}')
print(f'Min words    : {min(chunk_words)}')
print(f'Max words    : {max(chunk_words)}')

## 4. Evaluation Dataset Analysis

In [ ]:
with open(EVAL_DIR / 'eval_dataset.json') as f:
    eval_data = json.load(f)

df_eval = pd.DataFrame(eval_data)
print(f'Total questions: {len(df_eval)}')
print('\nCategory counts:')
print(df_eval['category'].value_counts())

fig, ax = plt.subplots(figsize=(8, 4))
df_eval['category'].value_counts().plot(kind='barh', ax=ax, color='#2563EB', alpha=0.8)
ax.set_xlabel('Count')
ax.set_title('Evaluation Dataset — Question Categories')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Baseline vs RAG Evaluation Results

In [ ]:
from config.settings import METRICS_DIR
import json

summary_path = METRICS_DIR / 'evaluation_summary.json'
if summary_path.exists():
    with open(summary_path) as f:
        summary = json.load(f)
    
    metrics = ['mean_recall@1', 'mean_recall@3', 'mean_recall@5', 'mrr',
               'token_f1', 'citation_accuracy', 'abstention_accuracy']
    labels = ['Recall@1', 'Recall@3', 'Recall@5', 'MRR', 'Token F1', 'Citation Acc', 'Abstention Acc']
    
    d_vals = [summary['dense_retrieval'][m] for m in metrics]
    t_vals = [summary['tfidf_retrieval'][m] for m in metrics]
    
    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - width/2, t_vals, width, label='TF-IDF Baseline', color='#93C5FD', alpha=0.9)
    ax.bar(x + width/2, d_vals, width, label='Dense RAG', color='#2563EB', alpha=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score')
    ax.set_title('TF-IDF Baseline vs Dense RAG — All Metrics')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print('Run evaluation first: python scripts/run_evaluation.py')